<a href="https://colab.research.google.com/github/faezesarlakifar/AllerTrans/blob/main/feature-extraction/5.%20ProtT5-embeddings%20(recombinant).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## The super helpful command of the [ProtTrans repository](https://github.com/agemagician/ProtTrans) is used to extract embedding vectors for recombinant proteins derived from reviewed [UniProt](https://www.uniprot.org/uniprotkb?query=%22recombinant+protein%22&facets=reviewed%3Atrue) entries in FASTA format. ❤

In [1]:
# @markdown configs
!git clone https://github.com/agemagician/ProtTrans.git
!pip install -q torch transformers sentencepiece h5py
!pip install -q transformers

Cloning into 'ProtTrans'...
remote: Enumerating objects: 1021, done.
remote: Counting objects: 100% (217/217), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 1021 (delta 180), reused 112 (delta 107), pack-reused 804 (from 1)
Receiving objects: 100% (1021/1021), 57.04 MiB | 10.39 MiB/s, done.
Resolving deltas: 100% (560/560), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.4 MB/s eta 0:00:00
   ━━

In [2]:
# @markdown import necessaries
import numpy as np
import pandas as pd
import h5py

In [4]:
# @markdown mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
input_path = '/content/drive/MyDrive/allergen-detection/fasta-files/'

In [6]:
output_path = 'embeddings/'

In [9]:
# @title Extract embeddings from positive recombinant data
input_file = input_path+"positive-recombinant-cleaned.fasta"

!python ProtTrans/Embedding/prott5_embedder.py --input $input_file --output embeddings/recombinant_positive/residue_embeddings.h5
!python ProtTrans/Embedding/prott5_embedder.py --input $input_file --output embeddings/recombinant_positive/protein_embeddings.h5 --per_protein 1

2025-04-02 20:57:47.507657: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743627467.529483   16261 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743627467.536161   16261 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-02 20:57:47.560838: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Using device: cuda:0
Loading: Rostlab/prot_t5_xl_half_uniref50-enc
You are using the default legacy behaviour of the 

In [10]:
# @title Extract embeddings from negative recombinant data
input_file = input_path+"negative-recombinant-cleaned.fasta"

!python ProtTrans/Embedding/prott5_embedder.py --input $input_file --output embeddings/recombinant_negative/residue_embeddings.h5
!python ProtTrans/Embedding/prott5_embedder.py --input $input_file --output embeddings/recombinant_negative/protein_embeddings.h5 --per_protein 1

2025-04-02 21:01:55.917478: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743627715.942411   17354 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743627715.949271   17354 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-02 21:01:55.971933: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Using device: cuda:0
Loading: Rostlab/prot_t5_xl_half_uniref50-enc
You are using the default legacy behaviour of the 

In [11]:
# @markdown Create the dataframe helper function
def make_df(embedding_path, label):
    f = h5py.File(embedding_path, 'r')
    keys = f.keys()
    df = pd.DataFrame()
    for key in keys:
        df = df.append([f[key]], ignore_index=True)
    labels = [label] * len(keys)
    df['Label'] = labels
    df['id'] = keys

    return df

In [15]:
# @markdown Create the dataframe helper function

import h5py
import pandas as pd

def make_df(embedding_path, label):
    f = h5py.File(embedding_path, 'r')
    keys = list(f.keys())
    data = []

    for key in keys:
        data.append(f[key][:])  # Append the array associated with each key

    # Convert the list of data into a DataFrame
    df = pd.DataFrame(data)

    # Add the label and id columns
    df['Label'] = [label] * len(keys)
    df['id'] = keys

    return df


## Save the recombinant allergen positive data as a .csv file

In [27]:
h5_file_name = "protein_embeddings.h5"
embedding_path = output_path+"recombinant_positive/"+h5_file_name
df_recombinant_positive = make_df(embedding_path, 1)
df_recombinant_positive.to_csv('recombinant_positive.csv')
df_recombinant_positive.head()

,0,1,2,3,4,5,6,7,8,9,...,1016,1017,1018,1019,1020,1021,1022,1023,Label,id
0,0.008218,-0.051670,0.006441,-0.004239,-0.009964,-0.005410,0.057669,-0.147232,0.009960,0.029998,...,-0.011841,-0.054573,0.063861,-0.008501,-0.047472,0.092393,0.050970,0.123839,1,sp|A0A0K2GUJ4|QCR7_DERPT
1,0.012904,0.035395,0.086558,0.000690,0.013910,0.033456,-0.036199,-0.042972,-0.001633,-0.016713,...,-0.016275,-0.038517,0.062925,0.056847,0.065689,0.012632,0.040655,0.026179,1,sp|A0A2H4HHY6|GLOX_ARTAN
2,0.005775,0.023953,0.020556,-0.036124,-0.054134,-0.049739,0.013464,-0.078874,0.018411,-0.075168,...,0.011675,-0.030831,-0.004466,0.001873,-0.070552,0.035383,-0.043133,-0.024689,1,sp|A0A7G2A2Z3|NLTP_MACIN
3,0.033499,0.060306,0.024112,0.005078,-0.019688,0.015504,-0.029353,-0.085128,-0.035343,0.000789,...,-0.005501,-0.040112,0.065727,-0.005701,0.012215,0.014424,0.050410,0.001870,1,sp|A0A8B0RBM2|ACES_BOMIG
4,0.023394,0.014168,0.015246,-0.018648,-0.049538,0.000927,-0.034505,-0.121647,-0.053618,-0.064015,...,-0.042028,-0.058717,0.005290,0.025396,-0.005006,0.019539,0.021814,0.006587,1,sp|A1KXI1|BLOT3_BLOTA


## Save the recombinant allergen negative data as a .csv file

In [28]:
h5_file_name = "protein_embeddings.h5"
embedding_path = output_path+"recombinant_negative/"+h5_file_name
df_recombinant_negative = make_df(embedding_path, 1)
df_recombinant_negative.to_csv('recombinant_negative.csv')
df_recombinant_negative.head()

,0,1,2,3,4,5,6,7,8,9,...,1016,1017,1018,1019,1020,1021,1022,1023,Label,id
0,0.013436,-0.039913,0.007061,0.062740,-0.043623,0.027801,-0.064310,-0.076424,-0.042505,0.047416,...,-0.012087,0.006945,0.060507,0.030127,0.027324,0.030014,0.090579,0.036666,1,sp|B5AJT3|VMP02_EULPE
1,0.002391,0.036716,0.082468,-0.056381,-0.088624,-0.018664,0.021318,-0.147814,-0.005651,0.012859,...,-0.062091,-0.091339,0.037551,-0.003304,-0.011090,0.068761,0.069404,-0.006826,1,sp|C0H691|SCR2_ACRMI
2,-0.024575,-0.095755,0.002481,0.109302,-0.036534,-0.009374,-0.049481,-0.128437,0.001373,0.050128,...,-0.040177,0.025762,-0.046901,-0.045144,-0.060919,0.020978,0.041273,0.085568,1,sp|C4QVB0|RRP36_KOMPG
3,-0.023355,-0.001341,-0.023053,-0.023025,-0.031279,0.057096,-0.068569,-0.111369,0.002457,-0.010418,...,0.009164,-0.030585,0.033365,-0.018288,-0.072767,-0.053643,-0.021789,0.043047,1,sp|C4QW04|LCL3_KOMPG
4,0.011145,-0.104022,-0.024329,0.082886,0.018704,0.011943,-0.029920,-0.020030,0.026281,-0.045234,...,-0.037953,-0.018827,0.072845,-0.002788,-0.043375,-0.054977,0.029767,0.034114,1,sp|C4QWJ4|MDM10_KOMPG


## Merge dataframes into a single dataframe

In [29]:
df_recombinant = pd.concat([df_recombinant_positive, df_recombinant_negative]).sample(frac=1).reset_index(drop=True)

In [30]:
df_recombinant.head()

,0,1,2,3,4,5,6,7,8,9,...,1016,1017,1018,1019,1020,1021,1022,1023,Label,id
0,0.027064,0.028032,0.058204,0.008597,-0.009560,0.013149,-0.082644,-0.094474,-0.011705,-0.038316,...,-0.004931,-0.059760,0.027508,-0.003489,-0.018631,0.032065,-0.008539,0.024221,1,sp|Q5GMY3|GMCH_MALSM
1,0.012791,-0.103621,0.030675,0.032476,0.003075,-0.013211,-0.050219,-0.049055,0.038538,-0.001868,...,-0.037868,-0.037373,0.018352,0.034478,0.029086,-0.062517,0.051323,0.070533,1,sp|C7E9V9|SCH34_STACH
2,0.011361,0.003533,0.031491,-0.010823,-0.007967,0.039780,-0.008649,-0.081571,0.010433,-0.078721,...,-0.014223,-0.025526,0.044710,0.008849,-0.005146,0.038961,0.054947,0.021287,1,sp|V5LU01|CEP01_AMBAR
3,-0.016785,0.081792,0.033216,-0.001997,0.003866,-0.014062,0.066917,-0.030769,-0.005011,-0.041625,...,-0.026331,-0.001025,0.035879,-0.026063,-0.004893,0.060487,0.012191,-0.028521,1,sp|P81295|PRR3_JUNAS
4,0.023460,-0.047014,0.034690,-0.069520,0.020334,0.020611,-0.002960,-0.141983,-0.001380,-0.018745,...,-0.018276,-0.014078,-0.006972,-0.112850,-0.017422,-0.069397,0.078808,0.060572,1,sp|Q66RP5|FABP_TYRPU


In [31]:
df_recombinant.to_csv('df_recombinant.csv')

In [32]:
len(df_recombinant)

84

In [33]:
file_path = "/content/drive/My Drive/allergen-detection/embeddings/protBERT-embeddings-with-id/df_recombinant.csv"

In [34]:
# @title Save Final CSV file to Drive
import os

parent_dir = os.path.dirname(file_path)
os.makedirs(parent_dir, exist_ok=True)

df_recombinant.to_csv(file_path, index=False)

print(f"CSV file saved to: {file_path}")


CSV file saved to: /content/drive/My Drive/allergen-detection/embeddings/protBERT-embeddings-with-id/df_recombinant.csv
